In [8]:
import requests
from bs4 import BeautifulSoup
import chinese_calendar
import json

print("日历取数环境已就绪")

日历取数环境已就绪


# HK public holidays

In [43]:
url = "https://www.1823.gov.hk/common/ical/en.json"
response = requests.get(url, timeout=30)

print("状态码：", response.status_code)

if response.status_code == 200:
    data = json.loads(response.content.decode("utf-8-sig"))
    print("数据类型：", type(data))
    print("最外层的键：", data.keys())
else:
    print("下载失败：", response.text[:200])

状态码： 200
数据类型： <class 'dict'>
最外层的键： dict_keys(['vcalendar'])


In [44]:
calendars = data["vcalendar"]
events = calendars[0]["vevent"]

print("Number of holiday records:", len(events))
print("First record:", events[0])

Number of holiday records: 51
First record: {'dtstart': ['20250101', {'value': 'DATE'}], 'dtend': ['20250102', {'value': 'DATE'}], 'dtstamp': '20250506T035044Z', 'transp': 'TRANSPARENT', 'uid': '20250101@1823.gov.hk', 'summary': 'The first day of January'}


In [45]:
first_event = events[0]

print("Date field:", first_event["dtstart"])
print("Holiday name:", first_event["summary"])

Date field: ['20250101', {'value': 'DATE'}]
Holiday name: The first day of January


In [46]:
def read_holiday(event):
    holiday_date = event["dtstart"][0]
    holiday_name = event["summary"]
    return {"date": holiday_date, "name": holiday_name}


assert read_holiday(first_event) == {
    "date": "20250101",
    "name": "The first day of January",
}

print(read_holiday(first_event))

{'date': '20250101', 'name': 'The first day of January'}


In [47]:
holidays_2025_2027 = []

for event in events:
    holiday = read_holiday(event)
    holidays_2025_2027.append(holiday)

print("Number of holidays:", len(holidays_2025_2027))
print("First holiday:", holidays_2025_2027[0])
print("Last holiday:", holidays_2025_2027[-1])

Number of holidays: 51
First holiday: {'date': '20250101', 'name': 'The first day of January'}
Last holiday: {'date': '20271227', 'name': 'The first weekday after Christmas Day'}


In [48]:
from pathlib import Path

print("Notebook working directory:", Path.cwd())

Notebook working directory: /Users/lvchris/Desktop/northbound_project/notebooks


In [49]:
raw_dir = Path.cwd().parent / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

raw_path = raw_dir / "hk_public_holidays_1823_2025_2027.json"
raw_path.write_bytes(response.content)

print("Saved raw file:", raw_path)
print("File size in bytes:", raw_path.stat().st_size)

Saved raw file: /Users/lvchris/Desktop/northbound_project/data/raw/hk_public_holidays_1823_2025_2027.json
File size in bytes: 15520


In [50]:
url_2023 = "https://www.gov.hk/en/about/abouthk/holiday/2023.htm"
response_2023 = requests.get(url_2023, timeout=30)

print("Status code:", response_2023.status_code)
print("HTML length:", len(response_2023.text))
print("Contains holiday table:", "Every Sunday" in response_2023.text)

Status code: 200
HTML length: 86583
Contains holiday table: True


In [51]:
html_path_2023 = raw_dir / "hk_public_holidays_2023.html"
html_path_2023.write_bytes(response_2023.content)

print("Saved raw HTML:", html_path_2023.relative_to(Path.cwd().parent))
print("File size in bytes:", html_path_2023.stat().st_size)

Saved raw HTML: data/raw/hk_public_holidays_2023.html
File size in bytes: 86771


In [52]:
soup_2023 = BeautifulSoup(html_path_2023.read_bytes(), "html.parser")
tables_2023 = soup_2023.find_all("table")

print("Number of tables:", len(tables_2023))

for table in tables_2023:
    if "Every Sunday" in table.get_text():
        holiday_table_2023 = table
        print("Rows in holiday table:", len(table.find_all("tr")))

Number of tables: 2
Rows in holiday table: 18


In [53]:
rows_2023 = holiday_table_2023.find_all("tr")
cells = rows_2023[1].find_all("td")

for cell in cells:
    print(cell.get_text(" ", strip=True))

The day following the first day of January
2 January
Monday


In [54]:
def read_holiday_row(row):
    cells = row.find_all("td")
    holiday_name = cells[0].get_text(" ", strip=True)
    date_text = " ".join(cells[1].get_text(" ", strip=True).split())
    return {"date_text": date_text, "name": holiday_name}


assert read_holiday_row(rows_2023[1]) == {
    "date_text": "2 January",
    "name": "The day following the first day of January",
}



In [55]:
print(read_holiday_row(rows_2023[1]))
print(repr(read_holiday_row(rows_2023[1])["date_text"]))
print(repr(read_holiday_row(rows_2023[1])["name"]))


{'date_text': '2 January', 'name': 'The day following the first day of January'}
'2 January'
'The day following the first day of January'


In [56]:
holidays_2023 = []

for row in rows_2023[1:]:
    holiday = read_holiday_row(row)
    holidays_2023.append(holiday)

print("Number of dated holidays:", len(holidays_2023))
print("First holiday:", holidays_2023[0])
print("Last holiday:", holidays_2023[-1])

Number of dated holidays: 17
First holiday: {'date_text': '2 January', 'name': 'The day following the first day of January'}
Last holiday: {'date_text': '26 December', 'name': 'The first weekday after Christmas Day'}


In [57]:
month_numbers = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12,
}

print(month_numbers["January"])
print(month_numbers["December"])

1
12


In [58]:
def format_holiday_date(date_text, year):
    day_text, month_name = date_text.split()
    day = int(day_text)
    month = month_numbers[month_name]
    return f"{year}{month:02d}{day:02d}"

print(format_holiday_date("2 January", 2023))

assert format_holiday_date("2 January", 2023) == "20230102"
assert format_holiday_date("26 December", 2023) == "20231226"

20230102


In [59]:
formatted_holidays_2023 = []

for holiday in holidays_2023:
    formatted_date = format_holiday_date(holiday["date_text"], 2023)
    formatted_holidays_2023.append({
        "date": formatted_date,
        "name": holiday["name"],
    })

print("Number of holidays:", len(formatted_holidays_2023))
print("First holiday:", formatted_holidays_2023[0])
print("Last holiday:", formatted_holidays_2023[-1])

Number of holidays: 17
First holiday: {'date': '20230102', 'name': 'The day following the first day of January'}
Last holiday: {'date': '20231226', 'name': 'The first weekday after Christmas Day'}


In [60]:
url_2024 = "https://www.gov.hk/en/about/abouthk/holiday/2024.htm"
response_2024 = requests.get(url_2024, timeout=30)

print("Status code:", response_2024.status_code)
print("HTML length:", len(response_2024.text))
print("Contains holiday table:", "Every Sunday" in response_2024.text)


Status code: 200
HTML length: 86971
Contains holiday table: True


In [61]:
html_path_2024 = raw_dir / "hk_public_holidays_2024.html"
html_path_2024.write_bytes(response_2024.content)

print("Saved raw HTML:", html_path_2024.relative_to(Path.cwd().parent))
print("File size in bytes:", html_path_2024.stat().st_size)

Saved raw HTML: data/raw/hk_public_holidays_2024.html
File size in bytes: 87159


In [62]:
soup_2024 = BeautifulSoup(html_path_2024.read_bytes(), "html.parser")
tables_2024 = soup_2024.find_all("table")

print("Number of tables:", len(tables_2024))

for table in tables_2024:
    if "Every Sunday" in table.get_text():
        holiday_table_2024 = table
        print("Rows in holiday table:", len(table.find_all("tr")))

Number of tables: 2
Rows in holiday table: 18


In [63]:
rows_2024 = holiday_table_2024.find_all("tr")
first_holiday_2024 = read_holiday_row(rows_2024[1])

assert first_holiday_2024 == {
    "date_text": "1 January",
    "name": "The first day of January",
}

print(first_holiday_2024)

{'date_text': '1 January', 'name': 'The first day of January'}


In [64]:
formatted_holidays_2024 = []

for row in rows_2024[1:]:
    holiday = read_holiday_row(row)
    formatted_date = format_holiday_date(holiday["date_text"], 2024)
    formatted_holidays_2024.append({
        "date": formatted_date,
        "name": holiday["name"],
    })

print("Number of holidays:", len(formatted_holidays_2024))
print("First holiday:", formatted_holidays_2024[0])
print("Last holiday:", formatted_holidays_2024[-1])

Number of holidays: 17
First holiday: {'date': '20240101', 'name': 'The first day of January'}
Last holiday: {'date': '20241226', 'name': 'The first weekday after Christmas Day'}


In [65]:
for year in range(2023, 2028):
    variable_name = f"formatted_holidays_{year}"

    if variable_name in globals():
        holidays = globals()[variable_name]
        print(year, "ready:", len(holidays), "holidays")
    else:
        print(year, "MISSING")

2023 ready: 17 holidays
2024 ready: 17 holidays
2025 MISSING
2026 MISSING
2027 MISSING


In [66]:
print("Variables currently in memory:")

for name in sorted(globals()):
    if any(str(year) in name for year in range(2025, 2028)):
        if not name.startswith("_"):
            value = globals()[name]

            try:
                size = len(value)
                print(name, "->", type(value).__name__, "length:", size)
            except TypeError:
                print(name, "->", type(value).__name__)


print("\nSaved raw HTML files:")

for path in sorted(raw_dir.glob("hk_public_holidays_*.html")):
    print(path.name)

Variables currently in memory:
holidays_2025_2027 -> list length: 51
html_path_2025 -> PosixPath
response_2025 -> Response
url_2025 -> str length: 52

Saved raw HTML files:
hk_public_holidays_2023.html
hk_public_holidays_2024.html
hk_public_holidays_2025.html


In [67]:
from collections import Counter

all_holidays = (
    formatted_holidays_2023
    + formatted_holidays_2024
    + holidays_2025_2027
)


assert all(
    "date" in holiday and "name" in holiday
    for holiday in all_holidays
)


all_holidays = sorted(
    all_holidays,
    key=lambda holiday: holiday["date"]
)


holiday_dates = [
    holiday["date"]
    for holiday in all_holidays
]

assert len(holiday_dates) == len(set(holiday_dates))


year_counts = Counter(
    holiday["date"][:4]
    for holiday in all_holidays
)

print("Total number of holidays:", len(all_holidays))
print("Holidays by year:", dict(sorted(year_counts.items())))
print("First holiday:", all_holidays[0])
print("Last holiday:", all_holidays[-1])

Total number of holidays: 85
Holidays by year: {'2023': 17, '2024': 17, '2025': 17, '2026': 17, '2027': 17}
First holiday: {'date': '20230102', 'name': 'The day following the first day of January'}
Last holiday: {'date': '20271227', 'name': 'The first weekday after Christmas Day'}


In [68]:
import csv
import json
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

json_path = processed_dir / "hk_public_holidays_2023_2027.json"
csv_path = processed_dir / "hk_public_holidays_2023_2027.csv"

# save JSON file
with json_path.open("w", encoding="utf-8") as file:
    json.dump(
        all_holidays,
        file,
        ensure_ascii=False,
        indent=2
    )

# save CSV file
with csv_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["date", "name"]
    )
    writer.writeheader()
    writer.writerows(all_holidays)

print("Saved JSON:", json_path)
print("JSON size:", json_path.stat().st_size, "bytes")
print("Saved CSV:", csv_path)
print("CSV size:", csv_path.stat().st_size, "bytes")

Saved JSON: ../data/processed/hk_public_holidays_2023_2027.json
JSON size: 6303 bytes
Saved CSV: ../data/processed/hk_public_holidays_2023_2027.csv
CSV size: 3167 bytes


In [69]:
# JSON
with json_path.open("r", encoding="utf-8") as file:
    saved_json = json.load(file)

# CSV
with csv_path.open("r", encoding="utf-8", newline="") as file:
    saved_csv = list(csv.DictReader(file))


assert len(saved_json) == 85
assert len(saved_csv) == 85

assert saved_json == all_holidays
assert saved_csv == all_holidays

print("JSON records:", len(saved_json))
print("CSV records:", len(saved_csv))
print("JSON and CSV match:", saved_json == saved_csv)
print("Validation passed.")

JSON records: 85
CSV records: 85
JSON and CSV match: True
Validation passed.


# Mainland holidays

In [ ]:
from datetime import date
import chinese_calendar

test_dates = [
    date(2023, 10, 1),  # national holiday
    date(2023, 10, 7),  # makeup workday
    date(2023, 10, 9),  # normal worday
]

for test_date in test_dates:
    holiday_detail = chinese_calendar.get_holiday_detail(test_date)

    print(
        test_date,
        "is_holiday:", chinese_calendar.is_holiday(test_date),
        "is_workday:", chinese_calendar.is_workday(test_date),
        "detail:", holiday_detail,
    )

2023-10-01 is_holiday: True is_workday: False detail: (True, 'National Day')
2023-10-07 is_holiday: False is_workday: True detail: (False, 'National Day')
2023-10-09 is_holiday: False is_workday: True detail: (False, None)


In [70]:
from datetime import date, timedelta

start_date = date(2023, 2, 6)
end_date = date(2026, 10, 31)

mainland_calendar = []
current_date = start_date

while current_date <= end_date:
    is_holiday, holiday_name = (
        chinese_calendar.get_holiday_detail(current_date)
    )

    mainland_calendar.append({
        "date": current_date.strftime("%Y%m%d"),
        "is_mainland_holiday": int(
            is_holiday and holiday_name is not None
        ),
        "is_mainland_makeup_workday": int(
            not is_holiday and holiday_name is not None
        ),
        "mainland_holiday_name": holiday_name or "",
    })

    current_date += timedelta(days=1)

print("Number of calendar days:", len(mainland_calendar))
print(
    "Mainland holiday days:",
    sum(row["is_mainland_holiday"] for row in mainland_calendar)
)
print(
    "Mainland makeup workdays:",
    sum(
        row["is_mainland_makeup_workday"]
        for row in mainland_calendar
    )
)
print("First day:", mainland_calendar[0])
print("Last day:", mainland_calendar[-1])

Number of calendar days: 1364
Mainland holiday days: 108
Mainland makeup workdays: 24
First day: {'date': '20230206', 'is_mainland_holiday': 0, 'is_mainland_makeup_workday': 0, 'mainland_holiday_name': ''}
Last day: {'date': '20261031', 'is_mainland_holiday': 0, 'is_mainland_makeup_workday': 0, 'mainland_holiday_name': ''}


In [71]:
mainland_by_date = {
    row["date"]: row
    for row in mainland_calendar
}

print("National Day holiday:")
print(mainland_by_date["20231001"])

print("\nMakeup workday:")
print(mainland_by_date["20231007"])

print("\nNormal workday:")
print(mainland_by_date["20231009"])

National Day holiday:
{'date': '20231001', 'is_mainland_holiday': 1, 'is_mainland_makeup_workday': 0, 'mainland_holiday_name': 'National Day'}

Makeup workday:
{'date': '20231007', 'is_mainland_holiday': 0, 'is_mainland_makeup_workday': 1, 'mainland_holiday_name': 'National Day'}

Normal workday:
{'date': '20231009', 'is_mainland_holiday': 0, 'is_mainland_makeup_workday': 0, 'mainland_holiday_name': ''}


In [72]:
assert mainland_by_date["20231001"]["is_mainland_holiday"] == 1

assert (
    mainland_by_date["20231007"]["is_mainland_makeup_workday"]
    == 1
)

assert (
    mainland_by_date["20231009"]["is_mainland_holiday"] == 0
    and mainland_by_date["20231009"]["is_mainland_makeup_workday"] == 0
)

print("Known-date tests passed.")

Known-date tests passed.


In [73]:
# Validate the mainland calendar structure and binary indicator values
dates = [
    row["date"]
    for row in mainland_calendar
]

holiday_values = {
    row["is_mainland_holiday"]
    for row in mainland_calendar
}

makeup_values = {
    row["is_mainland_makeup_workday"]
    for row in mainland_calendar
}

print("Number of rows:", len(mainland_calendar))
print("Number of unique dates:", len(set(dates)))
print("Holiday values:", holiday_values)
print("Makeup workday values:", makeup_values)

assert len(mainland_calendar) == 1364
assert len(set(dates)) == 1364
assert holiday_values <= {0, 1}
assert makeup_values <= {0, 1}

print("Calendar structure tests passed.")

Number of rows: 1364
Number of unique dates: 1364
Holiday values: {0, 1}
Makeup workday values: {0, 1}
Calendar structure tests passed.


In [74]:
# Count mainland holidays and makeup workdays by year

from collections import Counter

holiday_counts_by_year = Counter()
makeup_counts_by_year = Counter()

for row in mainland_calendar:
    year = row["date"][:4]

    holiday_counts_by_year[year] += (
        row["is_mainland_holiday"]
    )

    makeup_counts_by_year[year] += (
        row["is_mainland_makeup_workday"]
    )

print("Holiday days by year:",dict(sorted(holiday_counts_by_year.items())))
print("Makeup workdays by year:",dict(sorted(makeup_counts_by_year.items())))

Holiday days by year: {'2023': 19, '2024': 28, '2025': 28, '2026': 33}
Makeup workdays by year: {'2023': 5, '2024': 8, '2025': 5, '2026': 6}


In [75]:
# Save the processed mainland calendar data

import csv
import json
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

mainland_json_path = (
    processed_dir / "mainland_calendar_2023_2026.json"
)

mainland_csv_path = (
    processed_dir / "mainland_calendar_2023_2026.csv"
)

# Save as JSON
with mainland_json_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        mainland_calendar,
        file,
        ensure_ascii=False,
        indent=2
    )

# Save as CSV
with mainland_csv_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "date",
            "is_mainland_holiday",
            "is_mainland_makeup_workday",
            "mainland_holiday_name",
        ]
    )

    writer.writeheader()
    writer.writerows(mainland_calendar)

print("Saved JSON:", mainland_json_path)
print("Saved CSV:", mainland_csv_path)
print("JSON size:", mainland_json_path.stat().st_size, "bytes")
print("CSV size:", mainland_csv_path.stat().st_size, "bytes")

Saved JSON: ../data/processed/mainland_calendar_2023_2026.json
Saved CSV: ../data/processed/mainland_calendar_2023_2026.csv
JSON size: 181897 bytes
CSV size: 22382 bytes


In [76]:
# Reload and validate the saved mainland calendar files

with mainland_json_path.open(
    "r",
    encoding="utf-8"
) as file:
    saved_mainland_json = json.load(file)

with mainland_csv_path.open(
    "r",
    encoding="utf-8",
    newline=""
) as file:
    saved_mainland_csv = list(csv.DictReader(file))

# Convert CSV indicator values from text to integers
normalized_mainland_csv = []

for row in saved_mainland_csv:
    normalized_mainland_csv.append({
        "date": row["date"],
        "is_mainland_holiday": int(
            row["is_mainland_holiday"]
        ),
        "is_mainland_makeup_workday": int(
            row["is_mainland_makeup_workday"]
        ),
        "mainland_holiday_name": (
            row["mainland_holiday_name"]
        ),
    })

assert len(saved_mainland_json) == 1364
assert len(normalized_mainland_csv) == 1364

assert saved_mainland_json == mainland_calendar
assert normalized_mainland_csv == mainland_calendar

print("JSON records:", len(saved_mainland_json))
print("CSV records:", len(normalized_mainland_csv))
print("JSON and CSV match:",saved_mainland_json == normalized_mainland_csv)
print("Saved mainland calendar validation passed.")

JSON records: 1364
CSV records: 1364
JSON and CSV match: True
Saved mainland calendar validation passed.
